In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from google.colab import files

DEST = "/content/drive/MyDrive/longformer_runs/run_paper_v1/data/"
os.makedirs(DEST, exist_ok=True)

print("Please select: train.json, valid.json, test.json")
uploaded = files.upload()   

for name in uploaded.keys():
    dst = os.path.join(DEST, name)
    print(f"Saving {name} → {dst}")
    !mv "{name}" "{dst}"

print("\nUpload complete. Files currently in data folder:")
print(os.listdir(DEST))


Mounted at /content/drive
Please select: train.json, valid.json, test.json


Saving valid.json to valid.json
Saving train.json to train.json
Saving test.json to test.json
Saving valid.json → /content/drive/MyDrive/longformer_runs/run_paper_v1/data/valid.json
Saving train.json → /content/drive/MyDrive/longformer_runs/run_paper_v1/data/train.json
Saving test.json → /content/drive/MyDrive/longformer_runs/run_paper_v1/data/test.json

Upload complete. Files currently in data folder:
['valid.json', 'train.json', 'test.json']


In [2]:
!pip install -q "transformers>=4.41" datasets evaluate scikit-learn shap pandas numpy wandb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [ ]:
import os, pandas as pd, numpy as np, torch

BASE_DIR    = "/content/drive/MyDrive/longformer_runs/run_paper_v1"
DATA_DIR    = f"{BASE_DIR}/data"
RESULTS_DIR = f"{BASE_DIR}/results_longformer"
SHAP_DIR    = f"{BASE_DIR}/shap_outputs"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)

print("DATA_DIR   :", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("SHAP_DIR   :", SHAP_DIR)

import wandb
wandb.login()
os.environ["WANDB_PROJECT"] = "longformer_cci_paper"


DATA_DIR   : /content/drive/MyDrive/longformer_runs/run_paper_v1/data
RESULTS_DIR: /content/drive/MyDrive/longformer_runs/run_paper_v1/results_longformer
SHAP_DIR   : /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: maabdullatif1 (maabdullatif1-king-faisal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
train_df = pd.read_json(os.path.join(DATA_DIR, "train.json"))
val_df   = pd.read_json(os.path.join(DATA_DIR, "valid.json"))
test_df  = pd.read_json(os.path.join(DATA_DIR, "test.json"))

cols = ["new_comment_raw", "new_code_raw", "label"]
train_df = train_df[cols].copy()
val_df   = val_df[cols].copy()
test_df  = test_df[cols].copy()

train_df["label"] = train_df["label"].astype(int)
val_df["label"]   = val_df["label"].astype(int)
test_df["label"]  = test_df["label"].astype(int)

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()


(8398, 3) (1034, 3) (1066, 3)


,new_comment_raw,new_code_raw,label
0,Parses the given JSON and returns either a JSO...,public static JSONElement parse(InputStrea...,1
1,Loads an image from a given image identifier.,public static byte[] getImageInBytes(Strin...,0
2,Creates a new com.yammer.metrics.core.Counter...,public static Counter newCounter(Class<?> ...,1
3,Derives a sample format corresponding to a giv...,private static Format getSampleFormat(Format...,0
4,Revert a rotation/rotation rate/ rotation acce...,public AngularCoordinates revert() {\n ...,1


In [5]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)

raw = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")
MAX_LEN = 1024

def preprocess_function(examples):
    return tokenizer(
        examples["new_code_raw"],
        examples["new_comment_raw"],
        truncation=True,
        padding=False,
        max_length=MAX_LEN
    )

tokenized = raw.map(
    preprocess_function,
    batched=True,
    remove_columns=[c for c in raw["train"].column_names if c not in ["label"]]
)

tokenized


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8398 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 8398
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1034
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1066
    })
})

In [ ]:
from transformers import DataCollatorWithPadding
import torch

base_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def longformer_collator(features):
    batch = base_collator(features)
    ga = torch.zeros_like(batch["attention_mask"])
    ga[:, 0] = 1  
    batch["global_attention_mask"] = ga
    return batch


In [7]:
import evaluate
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import AutoModelForSequenceClassification

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "f1":        f1_score(labels, preds, zero_division=0),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    "allenai/longformer-base-4096",
    num_labels=2
)
model.gradient_checkpointing_enable()
model.config.use_cache = False


pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=RESULTS_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    eval_accumulation_steps=32,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="wandb",
    run_name="longformer-paper-v1",
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=longformer_collator,
    compute_metrics=compute_metrics
)


model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

/tmp/ipython-input-3916898517.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

test_results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Test metrics:", test_results)

import json
with open(os.path.join(RESULTS_DIR, "test_metrics.json"), "w") as f:
    json.dump(test_results, f, indent=2)

best_ckpt = trainer.state.best_model_checkpoint
print("Best checkpoint:", best_ckpt)

with open(os.path.join(RESULTS_DIR, "BEST_CHECKPOINT.txt"), "w") as f:
    f.write(best_ckpt + "\n")

wandb.log({f"test_{k}": v for k, v in test_results.items()})


Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.626700,0.297721,0.879110,1.000000,0.758221,0.862486
2,0.426200,0.302668,0.874275,0.886228,0.858801,0.872299
3,0.374900,0.283810,0.882012,0.934066,0.822050,0.874486
4,0.320600,0.313842,0.875242,0.902490,0.841393,0.870871
5,0.279300,0.389653,0.861702,0.854167,0.872340,0.863158


Test metrics: {'eval_loss': 0.30764836072921753, 'eval_accuracy': 0.8677298311444653, 'eval_precision': 0.9474885844748858, 'eval_recall': 0.7786116322701688, 'eval_f1': 0.854788877445932, 'eval_runtime': 92.4199, 'eval_samples_per_second': 11.534, 'eval_steps_per_second': 11.534, 'epoch': 5.0}
Best checkpoint: /content/drive/MyDrive/longformer_runs/run_paper_v1/results_longformer/checkpoint-1575
